In [7]:
import pandas as pd

# 1. Load dataset
df = pd.read_csv("jobs_clean.csv")

# 2. Clean missing values and format experience values
df_clean = df.dropna(
    subset=["minimumYearsExperience", "average_salary"]
).copy()
df_clean["minimumYearsExperience"] = df_clean[
    "minimumYearsExperience"
].astype(int)

# 3. Aggregate mean and median salary for 0-15 years of experience
grouped = (
    df_clean[df_clean["minimumYearsExperience"] <= 15]
    .groupby("minimumYearsExperience")
    .agg(
        avg_salary=("average_salary", "mean"),
        median_salary=("average_salary", "median"),
    )
    .reset_index()
)

# 4. Calculate year-over-year SGD and percentage changes
grouped["avg_sgd_change"] = grouped["avg_salary"].diff()
grouped["avg_pct_change"] = grouped["avg_salary"].pct_change() * 100
grouped["median_sgd_change"] = grouped["median_salary"].diff()
grouped["median_pct_change"] = grouped["median_salary"].pct_change() * 100

# 5. Format into printable display table
table_display = pd.DataFrame({
    "Minimum Experience": grouped["minimumYearsExperience"].apply(
        lambda x: f"{x} Year{'s' if x != 1 else ''}"
    ),
    "Average Salary": grouped["avg_salary"].apply(lambda x: f"SGD {x:,.2f}"),
    "Average Change vs Prior Year": grouped.apply(
        lambda r: (
            "-"
            if pd.isna(r["avg_sgd_change"])
            else f"+SGD {r['avg_sgd_change']:,.2f} (+{r['avg_pct_change']:.2f}%)"
        ),
        axis=1,
    ),
    "Position Levels Median": grouped["median_salary"].apply(
        lambda x: f"SGD {x:,.2f}"
    ),
    "Median Change vs Prior Year": grouped.apply(
        lambda r: (
            "-"
            if pd.isna(r["median_sgd_change"])
            else f"{'+' if r['median_sgd_change'] >= 0 else '-'}SGD {abs(r['median_sgd_change']):,.2f} ({'+' if r['median_pct_change'] >= 0 else ''}{r['median_pct_change']:.2f}%)"
        ),
        axis=1,
    ),
})

# Display table
print(table_display.to_string(index=False))

Minimum Experience Average Salary Average Change vs Prior Year Position Levels Median Median Change vs Prior Year
           0 Years   SGD 3,526.56                            -           SGD 3,100.00                           -
            1 Year   SGD 4,583.05      +SGD 1,056.48 (+29.96%)           SGD 4,000.00       +SGD 900.00 (+29.03%)
           2 Years   SGD 4,921.77         +SGD 338.72 (+7.39%)           SGD 4,500.00       +SGD 500.00 (+12.50%)
           3 Years   SGD 6,412.84      +SGD 1,491.08 (+30.30%)           SGD 6,000.00     +SGD 1,500.00 (+33.33%)
           4 Years   SGD 7,297.55        +SGD 884.71 (+13.80%)           SGD 7,000.00     +SGD 1,000.00 (+16.67%)
           5 Years   SGD 8,370.96      +SGD 1,073.41 (+14.71%)           SGD 8,000.00     +SGD 1,000.00 (+14.29%)
           6 Years   SGD 8,771.68         +SGD 400.71 (+4.79%)           SGD 8,250.00        +SGD 250.00 (+3.12%)
           7 Years  SGD 10,127.37      +SGD 1,355.70 (+15.46%)           SGD 9,000.00   